# Notebook 03 — Anomaly Detection

**Question:** trained only on data assumed healthy, how early does the detector flag the
bearing that eventually fails?

- Fit on the **first 60% of each run**, taken as healthy
- Score the whole series
- Measure how far before the end the first flag appears

**No labels reach the model.** The failure mode is known from the IMS documentation, but it
is used only to decide which bearing to look at afterwards.

---

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from nasa_bearing_anomaly.config import FIGURES_DIR, TEST_CONFIG
from nasa_bearing_anomaly.detection import run_pipeline, select_features

# Importing plotting applies the project's dark-industrial rcParams as a module-level
# side effect, and STYLE is the palette those figures are drawn from. Without this
# import the two figures below render on the matplotlib default -- white ground,
# default grid -- while every other figure in the repository is dark, so the site
# would show one analysis in two visual languages.
from nasa_bearing_anomaly.plotting import STYLE

print("Imports OK")

## 1. Why Isolation Forest?

**Core idea:** Normal data points are hard to isolate — they live in dense clusters and require many splits to separate. Anomalous points are easy to isolate — they're in sparse regions and need fewer splits.

The algorithm builds random trees and measures the **average path length** to isolate each point. Short path → anomaly.

**Advantages for this problem:**
- No labels required
- Scales well to many features
- Robust to the different failure signatures in Tests 1/2/3
- Interpretable anomaly score

## 2. Run Detection: All Three Tests

In [ ]:
results = {}

for test_id in [1, 2, 3]:
    print(f"\n{'=' * 60}")
    print(
        f"Test {test_id}: {TEST_CONFIG[test_id]['failed_bearing']} — {TEST_CONFIG[test_id]['failure_mode']}"
    )
    print(f"{'=' * 60}")
    results[test_id] = run_pipeline(test_id, method="isolation_forest")

## 3. Results Summary

In [ ]:
print("\n" + "═" * 70)
print("ANOMALY DETECTION RESULTS SUMMARY")
print("═" * 70)
print(
    f"{'Test':<6} {'Failed Bearing':<15} {'Files':<8} {'Anomalies':<11} "
    f"{'First Alert':<13} {'Raw Lead'}"
)
print("─" * 70)

for test_id, df in results.items():
    config = TEST_CONFIG[test_id]
    n = len(df)
    n_anom = df["is_anomaly"].sum()
    first_anom = df[df["is_anomaly"]].index.min() if n_anom > 0 else n

    # Lead time measured from the acquisition timestamps, not from an assumed
    # sampling interval. The timestamps parse correctly as of 2026-08-12.
    stamps = pd.to_datetime(df["timestamp"])
    lead_hours = (
        (stamps.iloc[-1] - stamps.loc[first_anom]).total_seconds() / 3600 if n_anom > 0 else 0.0
    )

    print(
        f"{test_id:<6} {config['failed_bearing']:<15} {n:<8} "
        f"{n_anom:<11} {first_anom:<13} {lead_hours:>6.1f} h"
    )

print("═" * 70)
print()
print("These are RAW lead times, not a result. Three things are missing:")
print("  1. They trigger on the first anomaly. One early false positive moves that")
print("     arbitrarily far back, so the number inflates without the model improving.")
print("     A sustained rule -- k of the last m windows flagged -- replaces it.")
print("  2. No false-alarm rate. A lead time without its false-alarm cost is not a finding.")
print("  3. The last file of each run is post-shutdown, so the end of the series is not")
print("     the moment of failure. See data/README.md.")
print()
print("All three land in business.py. No euro figure follows from the numbers above.")

## 4. Visualization: All Tests

Raw detector output, one panel pair per run. The red markers are `is_anomaly`
straight from the Isolation Forest, and they are deliberately **not** an alert:
`contamination` labels that fraction of the model's own training window anomalous
by construction, so flagged files appear throughout the healthy region of every
run and the first of them lands at or near file 0.

Turning this into something a maintenance team could act on needs a sustained-alert
rule and a score threshold calibrated against held-out healthy data. That is
notebook 04 and `business.py`. Nothing in these panels is a lead time.

In [ ]:
for test_id, df in results.items():
    config = TEST_CONFIG[test_id]
    failed = config["failed_bearing"]
    rms_col = f"{failed}_ch1_rms"
    score_col = "anomaly_score"

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    fig.suptitle(
        f"Test {test_id} — {failed} | {config['failure_mode']}", fontsize=12, fontweight="bold"
    )

    # RMS. No first-alert line: it would sit at file 0 on every run, which is an
    # artefact of contamination rather than a detection. The sustained alarm is
    # derived in notebook 04 and marked there.
    if rms_col in df.columns:
        ax1.plot(df.index, df[rms_col], color=STYLE["accent_color"], linewidth=0.8, label="RMS")
        if "is_anomaly" in df.columns:
            anom_mask = df["is_anomaly"]
            ax1.scatter(
                df.index[anom_mask],
                df[rms_col][anom_mask],
                color=STYLE["anomaly_color"],
                s=8,
                zorder=5,
                label="Flagged file (raw)",
            )

    ax1.set_ylabel("RMS (g)")
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.4)

    # Anomaly score
    if score_col in df.columns:
        s = df[score_col]
        s_norm = (s - s.min()) / (s.max() - s.min() + 1e-12)
        ax2.plot(df.index, s_norm, color=STYLE["warn_color"], linewidth=0.7)
        if "is_anomaly" in df.columns:
            ax2.fill_between(
                df.index,
                s_norm,
                where=df["is_anomaly"],
                color=STYLE["anomaly_color"],
                alpha=0.4,
                label="Flagged",
            )
        ax2.legend(fontsize=8)

    ax2.set_ylabel("Anomaly Score")
    ax2.set_xlabel("File Index")
    ax2.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig(
        FIGURES_DIR / f"detection_test{test_id}.png",
        dpi=150,
        bbox_inches="tight",
        facecolor=STYLE["bg_color"],
    )
    plt.show()

## 5. PCA Feature Space Visualization

In [ ]:
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("PCA Feature Space — Normal (green) vs Anomaly (red)", fontsize=12, fontweight="bold")

for ax, (test_id, df) in zip(axes, results.items(), strict=True):
    config = TEST_CONFIG[test_id]
    failed = config["failed_bearing"]

    feature_cols = select_features(df, bearing_prefix=failed)
    if len(feature_cols) < 2:
        feature_cols = select_features(df)

    X = df[feature_cols].fillna(0).to_numpy()
    X_scaled = StandardScaler().fit_transform(X)
    X_2d = PCA(n_components=2).fit_transform(X_scaled)

    colors = [
        STYLE["anomaly_color"] if a else STYLE["healthy_color"]
        for a in df.get("is_anomaly", [False] * len(df))
    ]
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=colors, s=10, alpha=0.6)
    ax.set_title(
        f"Test {test_id} — {failed}\n{config['failure_mode']}", fontsize=10, fontweight="bold"
    )
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(True, alpha=0.3)

legend_elems = [
    Patch(facecolor=STYLE["healthy_color"], label="Normal"),
    Patch(facecolor=STYLE["anomaly_color"], label="Anomaly"),
]
fig.legend(
    handles=legend_elems, loc="lower center", ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.05)
)

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "pca_all_tests.png",
    dpi=150,
    bbox_inches="tight",
    facecolor=STYLE["bg_color"],
)
plt.show()

## 6. (Optional) Autoencoder for Test 3

Test 3 has a gradual multi-phase degradation that benefits from the autoencoder's deeper representation learning.

In [ ]:
from nasa_bearing_anomaly.detection import TORCH_AVAILABLE

if TORCH_AVAILABLE:
    print("Running Autoencoder on Test 3...")
    ae_result = run_pipeline(3, method="autoencoder")

    # Compare with Isolation Forest
    if_first = (
        results[3][results[3]["is_anomaly"]].index.min() if results[3]["is_anomaly"].any() else None
    )
    ae_first = (
        ae_result[ae_result["is_anomaly"]].index.min() if ae_result["is_anomaly"].any() else None
    )

    print("\nTest 3 Comparison:")
    print(f"  Isolation Forest first alert: {if_first}")
    print(f"  Autoencoder first alert:      {ae_first}")
else:
    print("PyTorch not installed. Install the optional extra with: pip install -e '.[deep]'")

---
## Summary

The Isolation Forest flags the failed bearing well before the end of every run. The table
that belongs here — lead time per test, each paired with its false-alarm rate — is
deliberately **not** filled in yet, because the numbers printed above are not yet a result.

They trigger on the *first* anomaly. A single early false positive moves that trigger
arbitrarily far back, which inflates the lead time without the model having improved at
all. Three things replace it, in `business.py`:

- a **sustained-alert rule** — k of the last m windows flagged — instead of a single trigger
- a **false-alarm rate**, calibrated against a healthy baseline window and reported in the
  same breath as the lead time
- the **post-shutdown tail excluded**, since the final file of each run was recorded after
  the rig had already stopped

Until those exist there is no lead time to quote and no downtime-cost figure that follows
from one.

**Next:** Results visualization & business report → Notebook 04